# 85 — Sales Call Prep Agent

## Goal
Build a **multi-agent crew** that **gathers company data**,
**summarizes account history**, and **generates a meeting brief**
before a sales call.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **CrewAI agents** | Specialized roles for research |
| **Task chaining** | Research → Summarize → Brief |
| **Tool use** | Agents call functions to fetch data |
| **Multi-agent** | 3 agents collaborating |

## Stack
- `crewai` — multi-agent orchestration
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew
from crewai.tools import tool

print("All imports OK")

## Step 1 — Simulated Data Sources

In production, these would hit real CRM, LinkedIn, and news APIs.

In [ ]:
# Simulated CRM data
CRM_DATA = {
    "NovaTech Solutions": {
        "account_id": "ACC-2024-0456",
        "industry": "Financial Technology",
        "size": "500-1000 employees",
        "revenue": "$85M ARR",
        "current_products": ["Analytics Pro (since 2022)", "Data Pipeline Starter (since 2023)"],
        "contract_value": "$120,000/year",
        "renewal_date": "2025-03-15",
        "primary_contact": "Sarah Kim, VP Engineering",
        "decision_maker": "James Park, CTO",
        "last_meeting": "2024-09-20 — Quarterly business review, discussed scaling needs",
        "open_tickets": 2,
        "nps_score": 7,
        "expansion_opportunities": ["Enterprise Analytics upgrade", "ML Platform add-on"],
        "notes": "Customer mentioned headcount growth planned for Q1 2025. Budget cycle starts January."
    }
}

# Simulated company profile
COMPANY_PROFILES = {
    "NovaTech Solutions": {
        "founded": 2018,
        "hq": "San Francisco, CA",
        "ceo": "Michael Torres",
        "recent_funding": "Series C, $50M (June 2024, led by Accel)",
        "competitors": ["Stripe", "Plaid", "Marqeta"],
        "tech_stack": ["Python", "Kubernetes", "PostgreSQL", "React"],
        "recent_news": [
            "Expanded to European market (Oct 2024)",
            "Hired new VP of Data (Sep 2024)",
            "Launched new payments API (Aug 2024)",
        ],
        "glassdoor_rating": 4.1
    }
}

print("Data sources loaded")

In [ ]:
# Define tools that agents can call
@tool
def lookup_crm(company_name: str) -> str:
    """Look up CRM data for a company including account history, contract details, and notes."""
    data = CRM_DATA.get(company_name)
    if data:
        return str(data)
    return f"No CRM data found for {company_name}"

@tool
def lookup_company_profile(company_name: str) -> str:
    """Look up public company information including funding, news, and tech stack."""
    data = COMPANY_PROFILES.get(company_name)
    if data:
        return str(data)
    return f"No company profile found for {company_name}"

print("Tools defined")

## Step 2 — Define the Crew

Three agents with distinct roles:
1. **Researcher** — gathers company and market data
2. **Account Analyst** — summarizes CRM history
3. **Brief Writer** — produces the final meeting prep

In [ ]:
researcher = Agent(
    role="Company Researcher",
    goal="Gather comprehensive company intelligence for sales meeting preparation",
    backstory="""You are a skilled sales intelligence researcher. You find
    relevant company information including recent news, funding, leadership
    changes, and competitive landscape to help the sales team prepare.""",
    tools=[lookup_company_profile],
    llm="ollama/qwen3.5:9b",
    verbose=True,
    max_iter=3
)

account_analyst = Agent(
    role="Account Analyst",
    goal="Analyze CRM data to summarize account health and identify opportunities",
    backstory="""You are an experienced account analyst who excels at reading
    CRM data and spotting expansion opportunities, risks, and patterns in
    customer behavior.""",
    tools=[lookup_crm],
    llm="ollama/qwen3.5:9b",
    verbose=True,
    max_iter=3
)

brief_writer = Agent(
    role="Meeting Brief Writer",
    goal="Create a concise, actionable meeting brief for the sales team",
    backstory="""You are a senior sales operations specialist who creates
    effective meeting briefs. You focus on key talking points, risks to
    address, and opportunities to pursue.""",
    llm="ollama/qwen3.5:9b",
    verbose=True,
    max_iter=3
)

print("3 agents defined")

In [ ]:
# Define tasks
company_name = "NovaTech Solutions"

research_task = Task(
    description=f"""Research {company_name} thoroughly.
    Use the lookup_company_profile tool to gather company data.
    Summarize: recent news, funding, leadership, competitive position, and tech stack.
    Highlight anything relevant for a sales conversation.""",
    expected_output="A structured company intelligence report with key findings",
    agent=researcher
)

account_task = Task(
    description=f"""Analyze the CRM data for {company_name}.
    Use the lookup_crm tool to get account details.
    Assess: account health (NPS, tickets), contract status, expansion opportunities,
    and any risks. Note the renewal timeline.""",
    expected_output="An account health summary with expansion opportunities and risks",
    agent=account_analyst
)

brief_task = Task(
    description=f"""Using the company research and account analysis, create a
    1-page meeting brief for a sales call with {company_name}.
    Include: 
    1. Company snapshot (1-2 sentences)
    2. Account status and health
    3. Key talking points (3-5 bullets)
    4. Risks to address
    5. Upsell/expansion opportunities
    6. Recommended ask/next steps""",
    expected_output="A concise, actionable 1-page meeting brief",
    agent=brief_writer,
    context=[research_task, account_task]
)

print("3 tasks defined")

In [ ]:
# Create and run the crew
crew = Crew(
    agents=[researcher, account_analyst, brief_writer],
    tasks=[research_task, account_task, brief_task],
    verbose=True
)

print("Crew assembled — running sales call prep...\n")
result = crew.kickoff()

print("\n" + "=" * 70)
print("MEETING BRIEF")
print("=" * 70)
print(result)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Role specialization** | 3 agents with distinct expertise |
| **Tool use** | Agents query CRM and company data |
| **Task context** | Brief writer uses research + analysis |
| **Sequential flow** | Research → Analyze → Write |

## Architecture
```
Company Researcher ──→ Company Intel Report
          ↓                       ↓
Account Analyst ──→ Account Health    ──→ Brief Writer ──→ Meeting Brief
       (CRM data)        Summary            (combines both)
```

## Extending This Project
- Integrate real CRM APIs (Salesforce, HubSpot)
- Add LinkedIn scraping for attendee profiles
- Competitive battle card generation
- Post-meeting action item tracker